# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Medical Cost Personal Dataset (Insurance)** yang bersumber dari Kaggle.

- **Sumber:** https://www.kaggle.com/datasets/mirichoi0218/insurance
- **Jumlah Data:** 1338 baris, 7 kolom
- **Task:** Regresi — memprediksi biaya asuransi kesehatan (`charges`)

### Deskripsi Fitur:
| Fitur | Tipe | Deskripsi |
|---|---|---|
| age | Numerik | Usia pemegang polis |
| sex | Kategorikal | Jenis kelamin (male/female) |
| bmi | Numerik | Body Mass Index |
| children | Numerik | Jumlah tanggungan |
| smoker | Kategorikal | Status perokok (yes/no) |
| region | Kategorikal | Wilayah (northeast/northwest/southeast/southwest) |
| charges | Numerik | **Target** — biaya asuransi (USD) |

# **2. Import Library**

In [ ]:
# Install kaggle jika belum ada
!pip install kaggle -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Library berhasil diimport!')
print(f'Pandas version: {pd.__version__}')
print(f'Numpy version: {np.__version__}')

# **3. Memuat Dataset**

In [ ]:
# Download dataset dari Kaggle
# Pastikan kaggle.json sudah diupload ke Kaggle Secrets
# Di Kaggle: Add-ons > Secrets > New Secret

import os
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

# Download dataset
api.dataset_download_files('mirichoi0218/insurance', path='./', unzip=True)
print('Dataset berhasil didownload!')

In [ ]:
# Load dataset
df = pd.read_csv('insurance.csv')

print('Shape dataset:', df.shape)
print('\n5 baris pertama:')
df.head()

In [ ]:
# Informasi dasar dataset
print('=== Info Dataset ===')
df.info()
print('\n=== Statistik Deskriptif ===')
df.describe()

# **4. Exploratory Data Analysis (EDA)**

In [ ]:
# 4.1 Cek Missing Values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# 4.2 Cek Duplikat
print(f'Jumlah duplikat: {df.duplicated().sum()}')
df_clean = df.drop_duplicates()
print(f'Shape setelah hapus duplikat: {df_clean.shape}')

In [ ]:
# 4.3 Distribusi Target Variable (charges)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['charges'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Distribusi Charges (Original)')
axes[0].set_xlabel('Charges (USD)')
axes[0].set_ylabel('Frekuensi')

axes[1].hist(np.log1p(df_clean['charges']), bins=50, color='coral', edgecolor='black')
axes[1].set_title('Distribusi Log(Charges)')
axes[1].set_xlabel('Log Charges')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('eda_charges_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Target variable memiliki distribusi right-skewed, perlu diperhatikan.')

In [ ]:
# 4.4 Distribusi Fitur Numerik
num_cols = ['age', 'bmi', 'children', 'charges']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df_clean[col], bins=30, color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('eda_numeric_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 4.5 Distribusi Fitur Kategorikal
cat_cols = ['sex', 'smoker', 'region']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(cat_cols):
    counts = df_clean[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_categorical_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 4.6 Pengaruh Smoker terhadap Charges
fig, ax = plt.subplots(figsize=(8, 5))
df_clean.boxplot(column='charges', by='smoker', ax=ax)
ax.set_title('Charges berdasarkan Status Perokok')
ax.set_xlabel('Smoker')
ax.set_ylabel('Charges (USD)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('eda_smoker_charges.png', dpi=100, bbox_inches='tight')
plt.show()

print('Rata-rata charges per kategori smoker:')
print(df_clean.groupby('smoker')['charges'].mean())

In [ ]:
# 4.7 Correlation Heatmap (Numerik)
df_encoded = df_clean.copy()
df_encoded['sex'] = df_encoded['sex'].map({'male': 1, 'female': 0})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})
df_encoded['region'] = df_encoded['region'].map({'northeast': 0, 'northwest': 1, 'southeast': 2, 'southwest': 3})

plt.figure(figsize=(8, 6))
corr = df_encoded.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nKorelasi dengan target (charges):')
print(corr['charges'].sort_values(ascending=False))

In [ ]:
# 4.8 Deteksi Outlier menggunakan IQR
def detect_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower) | (data[col] > upper)]
    return outliers, lower, upper

for col in ['age', 'bmi', 'charges']:
    outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    print(f'{col}: {len(outliers)} outlier | lower={lower:.2f}, upper={upper:.2f}')

# **5. Data Preprocessing**

In [ ]:
# 5.1 Hapus Duplikat
df_processed = df.copy()
before = len(df_processed)
df_processed = df_processed.drop_duplicates()
after = len(df_processed)
print(f'Duplikat dihapus: {before - after} baris')
print(f'Shape setelah hapus duplikat: {df_processed.shape}')

In [ ]:
# 5.2 Encoding Fitur Kategorikal
# sex: binary encoding
df_processed['sex'] = df_processed['sex'].map({'male': 1, 'female': 0})

# smoker: binary encoding
df_processed['smoker'] = df_processed['smoker'].map({'yes': 1, 'no': 0})

# region: one-hot encoding
df_processed = pd.get_dummies(df_processed, columns=['region'], drop_first=False)

print('Setelah encoding:')
print(df_processed.head())
print(f'\nShape: {df_processed.shape}')
print(f'Kolom: {list(df_processed.columns)}')

In [ ]:
# 5.3 Penanganan Outlier pada BMI menggunakan capping
Q1_bmi = df_processed['bmi'].quantile(0.25)
Q3_bmi = df_processed['bmi'].quantile(0.75)
IQR_bmi = Q3_bmi - Q1_bmi
lower_bmi = Q1_bmi - 1.5 * IQR_bmi
upper_bmi = Q3_bmi + 1.5 * IQR_bmi

df_processed['bmi'] = df_processed['bmi'].clip(lower=lower_bmi, upper=upper_bmi)
print(f'BMI setelah capping: min={df_processed["bmi"].min():.2f}, max={df_processed["bmi"].max():.2f}')

In [ ]:
# 5.4 Feature & Target Split
X = df_processed.drop('charges', axis=1)
y = df_processed['charges']

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')
print(f'\nFitur: {list(X.columns)}')

In [ ]:
# 5.5 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

In [ ]:
# 5.6 Normalisasi Fitur Numerik
scaler = StandardScaler()
num_features = ['age', 'bmi', 'children']

X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print('Setelah StandardScaler:')
print(X_train[num_features].describe().round(3))

In [ ]:
# 5.7 Simpan hasil preprocessing
import os
os.makedirs('insurance_preprocessing', exist_ok=True)

# Gabungkan X_train + y_train
train_data = X_train.copy()
train_data['charges'] = y_train.values

# Gabungkan X_test + y_test
test_data = X_test.copy()
test_data['charges'] = y_test.values

train_data.to_csv('insurance_preprocessing/train.csv', index=False)
test_data.to_csv('insurance_preprocessing/test.csv', index=False)

print('Dataset hasil preprocessing berhasil disimpan!')
print(f'Train: {train_data.shape}')
print(f'Test: {test_data.shape}')

In [ ]:
# 5.8 Verifikasi hasil akhir
print('=== Dataset Siap Latih ===')
print(f'Fitur: {list(X_train.columns)}')
print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')
print(f'\nTrain head:')
pd.read_csv('insurance_preprocessing/train.csv').head()